# 14 · Pilot-aided and decision-directed carrier recovery

Blind CPR ([13](13-blind-carrier-phase-recovery.ipynb)) is free of overhead but leaves a 90° ambiguity
and can slip.  Spending a little capacity on **known signals** removes both
problems:

1. **pilot symbols**: known symbols at known positions;
2. a static **data-aided** rotation;
3. a **decision-directed PLL**, which tracks from its own decisions;
4. a **pilot tone**: a CW line that carries the carrier phase itself.

All are method objects for the same `estimate_carrier_phase` /
`correct_carrier_phase` pair.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.recovery import BPS, PLL, DataAided, PilotAided, PilotTone

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
RS = 32e9
N = 2**16


def host(a):
    return to_device(a, "cpu")


def score(y):
    """SNR after resolving any 90-degree ambiguity against the reference."""
    return ck.metrics.snr(ck.recovery.resolve_phase_ambiguity(y))


tx = ck.generate(C16, num_symbols=N, symbol_rate=RS, rng=1, device=DEVICE)


def received(linewidth, esn0_db, rng=2):
    return ck.impairments.apply_awgn(
        ck.impairments.apply_phase_noise(tx, linewidth=linewidth, rng=rng),
        esn0_db=esn0_db,
        rng=rng + 1,
    )

## 1. Pilot symbols

With a known symbol `s_k` at position `k`, `∠(r_k · s_k*)` is a direct,
unambiguous phase measurement.  `PilotAided(indices, values)` measures at
every pilot, unwraps and interpolates in between.  There is no M-th power,
no decision, no ambiguity, and no slip unless the phase moves by more than
`π` between two pilots.  The cost is the overhead (`1/spacing` of the
symbols) and the noise of a measurement from a single symbol.

At a 10 MHz linewidth and Es/N0 = 14 dB, BPS slips whatever its block size;
pilots every 16 symbols hold:

In [ ]:
rx = received(10e6, 14)
for b in (16, 32, 64):
    print(
        f"BPS block {b:2d}:          SNR {score(ck.recovery.correct_carrier_phase(rx, BPS(block_size=b))):6.2f} dB"
    )
ref = host(tx.reference.symbols)
for spacing in (8, 16, 32, 64):
    idx = np.arange(0, N, spacing)
    y = ck.recovery.correct_carrier_phase(rx, PilotAided(indices=idx, values=ref[idx]))
    print(
        f"pilots every {spacing:2d} ({100 / spacing:4.1f} % overhead): SNR {ck.metrics.snr(y):6.2f} dB, no ambiguity to resolve"
    )

Each pilot measures the phase with the noise of one symbol, so pilot-only
CPR stays a few dB below the Es/N0 where BPS works.  The noise can be
reduced by smoothing the pilot phases with the Wiener prior of a random-walk
phase (`smooth_phase_wiener`), which is the optimal linear smoother for
this model.  In practice receivers combine both worlds: BPS for accuracy,
pilots to resolve its ambiguity and catch its slips.

In [ ]:
rx = received(2e6, 14)
idx = np.arange(0, N, 32)
est = ck.recovery.estimate_carrier_phase(rx, PilotAided(indices=idx, values=ref[idx]))
smoothed = ck.recovery.smooth_phase_wiener(
    est.pilot_phase,
    linewidth=2e6,
    sampling_rate=RS / 32,
    measurement_variance=10 ** (-14 / 10) / 2,
)
trajectory = np.interp(np.arange(N), idx, host(smoothed))
print(
    f"2 MHz, 14 dB: BPS {score(ck.recovery.correct_carrier_phase(rx, BPS())):.2f} dB, "
    f"pilots {ck.metrics.snr(ck.recovery.correct_carrier_phase(rx, est)):.2f} dB, "
    f"Wiener-smoothed pilots {ck.metrics.snr(ck.recovery.correct_carrier_phase(rx, trajectory)):.2f} dB"
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(idx[:400], host(est.pilot_phase)[:400], ".", label="pilot measurements")
ax.plot(idx[:400], host(smoothed)[:400], label="Wiener-smoothed")
ax.set_xlabel("Symbol")
ax.set_ylabel("Phase [rad]")
ax.legend()
plt.show()

## 2. A static data-aided rotation

Blind equalizers (CMA, RDE, [17](17-blind-equalization.ipynb)) are rotation invariant and leave each
output channel with an arbitrary constant phase, not limited to multiples
of 90°.  `DataAided()` fits one phase per channel against known symbols (by
default the `Signal`'s reference), `θ = ∠ Σ y · s*`.

In [ ]:
rotated = tx.replace(samples=tx.samples * np.complex64(np.exp(1j * 0.7)))
rotated = ck.impairments.apply_awgn(rotated, esn0_db=20, rng=5)
print(
    f"before {ck.metrics.snr(rotated):6.2f} dB, after DataAided {ck.metrics.snr(ck.recovery.correct_carrier_phase(rotated, DataAided())):6.2f} dB"
)

## 3. A decision-directed PLL

A phase-locked loop tracks symbol by symbol: derotate, decide, measure the
phase error against the decision, and update.  `PLL(normalized_bandwidth=B)` sets a
critically damped second-order loop with normalized bandwidth `B` (a
fraction of the symbol rate).  It has no block structure and no delay,
which makes it the CPR of choice *inside* an adaptive equalizer
([16](16-adaptive-equalization.ipynb)), but it needs reliable decisions: the
bandwidth must follow the phase noise without letting decision errors
through, and a loop that loses lock slips.

In [ ]:
bandwidths = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
fig, ax = plt.subplots(figsize=(8, 4))
for lw in (100e3, 1e6):
    r = received(lw, 18, rng=6)
    ax.semilogx(
        bandwidths,
        [
            score(ck.recovery.correct_carrier_phase(r, PLL(normalized_bandwidth=b)))
            for b in bandwidths
        ],
        "o-",
        label=f"PLL, Δν = {lw / 1e6:g} MHz",
    )
    ax.axhline(
        score(ck.recovery.correct_carrier_phase(r, BPS())),
        ls=":",
        color=ax.lines[-1].get_color(),
        label=f"BPS, Δν = {lw / 1e6:g} MHz",
    )
ax.set_ylim(0, 20)
ax.set_xlabel("Loop bandwidth [fraction of the symbol rate]")
ax.set_ylabel("SNR after CPR [dB]")
ax.set_title("16-QAM, Es/N0 = 18 dB")
ax.legend()
plt.show()

## 4. A pilot tone

A CW tone transmitted next to the data band ([02](02-spectral-analysis.ipynb) · section 5) passes
through the same lasers, so its phase *is* the carrier phase: frequency
offset and phase noise together, with no decisions and no M-th power.
`PilotTone(frequency, bandwidth)` isolates the tone with a zero-phase
spectral window of half-width `bandwidth` and returns its phase.  It works on
the oversampled waveform before the matched filter (which removes the
tone).  The window bandwidth trades phase-noise tracking (wider) against
the noise that enters with the tone (narrower).

In [ ]:
tx2 = ck.generate(
    C16,
    num_symbols=N,
    symbol_rate=RS,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=7,
    device=DEVICE,
)
with_tone = ck.spectral.add_pilot_tone(tx2, frequency=20e9, power_ratio_db=-12)
r2 = ck.spectral.shift_frequency(with_tone, frequency=50e6)
r2 = ck.impairments.apply_awgn(
    ck.impairments.apply_phase_noise(r2, linewidth=2e6, rng=8), esn0_db=14, rng=9
)

for bw in (10e6, 50e6, 200e6, 800e6):
    method = PilotTone(frequency=20e9, bandwidth=bw, search_band=200e6)
    y = ck.multirate.decimate_to_symbol_rate(
        ck.filtering.matched_filter(ck.recovery.correct_carrier_phase(r2, method))
    )
    y = ck.recovery.correct_carrier_phase(
        y, DataAided()
    )  # the tone's own constant phase
    print(f"tone window ±{bw / 1e6:4.0f} MHz: SNR {ck.metrics.snr(y):5.2f} dB")
ck.plotting.plot_pilot_tone_phase_estimate(
    ck.recovery.estimate_carrier_phase(
        r2, PilotTone(frequency=20e9, bandwidth=200e6, search_band=200e6)
    ),
    r2,
    method=PilotTone(frequency=20e9, bandwidth=200e6, search_band=200e6),
    show=True,
)

The tone removed the 50 MHz frequency offset and the phase noise in one
step, without a separate FOE.  The price is the tone power (here 12 dB below
the data, about 6 % of the transmit power) and a guard band around it.
`PilotTones` combines several tones by maximum-ratio combining, e.g. one per
polarization.

**Summary**

| method | ambiguity | slips | overhead | needs |
|---|---|---|---|---|
| `BPS` (blind) | 90° | possible | none | moderate SNR and `Δν·T` |
| `PilotAided` | none | no (below π per pilot gap) | 1/spacing | pilot positions |
| `DataAided` | none | n/a (static) | known symbols | a constant phase |
| `PLL` | 90° | if it loses lock | none | reliable decisions |
| `PilotTone` | none | no | tone power, guard band | a tone in the spectrum |

**Next:** [15 · Linear channel equalization](15-linear-channel-equalization.ipynb).